# 14 — Agent1 integrated with V2
Run locally after copying the patch files into the project. Restart the notebook kernel after replacing Python modules. Release notebook 13's GPU model first by restarting that kernel and leaving it idle.

This connects the real saved classifier, router, analyzer, argument collection, mock order tool, intent-based knowledge store, and V2 response generator. The analyzer uses the **base model with the adapter temporarily disabled**; V2 is used for final responses. Only one Llama model is loaded into GPU memory. No Ollama service is required for this notebook.

The order service remains a mock with fixed historical dates. These are demonstration records, not live orders. Knowledge selection remains intent-based; vector retrieval is not introduced here.

In [ ]:
from pathlib import Path
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open the notebook inside your Agent1 project.")
sys.path.insert(0, str(project_root / "src"))
adapter_dir = project_root / "models/adapter_v2"
knowledge_path = project_root / "data/knowledge/demo_retail_knowledge_v1.json"
# Set this relative path only if you have multiple saved classifier files.
CLASSIFIER_PATH = None
if CLASSIFIER_PATH is None:
    candidates = sorted({*project_root.joinpath("models").rglob("*.joblib"), *project_root.joinpath("models").rglob("*.pkl")})
    if len(candidates) != 1:
        print("Saved model candidates:")
        for path in candidates: print(path.relative_to(project_root))
        raise ValueError("Set CLASSIFIER_PATH to your saved ensemble's relative path, then rerun this cell. No retraining is needed.")
    classifier_path = candidates[0]
else:
    classifier_path = project_root / CLASSIFIER_PATH
print("Classifier:", classifier_path)
print("Adapter:", adapter_dir)

In [ ]:
from agent1.classification.ensemble import IntentClassifierEnsemble
from agent1.routing.intent_router import IntentRouter
from agent1.routing.intent_mapping import INTENT_TO_ACTION
from agent1.analysis.request_analyzer import RequestAnalyzer
from agent1.conversation.context_builder import ContextBuilder
from agent1.rag.knowledge import KnowledgeStore
from agent1.generation.response_generator import ResponseGenerator
from agent1.generation.response_service import ResponseService
from agent1.agent import Agent

# Load only your own trusted saved classifier.
ensemble = IntentClassifierEnsemble.load(classifier_path)
# Leave the 4 GB GPU for Llama; sentence embeddings run on CPU.
ensemble.semantic_classifier.device = "cpu"
ensemble.semantic_classifier.embedding_model = None
knowledge = KnowledgeStore(knowledge_path)
generator = ResponseGenerator(adapter_dir)
responses = ResponseService(generator, knowledge)

def analyzer_llm(system_prompt, user_prompt):
    # V2 was trained to answer grounded questions, not to emit ActionPlan JSON.
    with generator.model.disable_adapter():
        return generator.generate([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ], max_new_tokens=512)

agent = Agent(
    ensemble_model=ensemble,
    router=IntentRouter(INTENT_TO_ACTION),
    analyzer=RequestAnalyzer(llm=analyzer_llm),
    context_builder=ContextBuilder(),
    response_service=responses,
)
print("Agent1 is ready. No training is performed.")

## Try the integrated flow
These calls use the real analyzer; plans are not preselected. Inspect the plan and response source printed after every turn. If the analyzer returns invalid JSON, its existing exception shows the raw output for debugging. These live LLM calls have not been executed in the delivered test environment.

In [ ]:
def chat(text):
    print("You:", text)
    answer = agent.handle_message(text)
    print("Agent1:", answer)
    print("Plan:", agent.last_action_plan.model_dump() if agent.last_action_plan else None)
    print("Response source:", responses.last_mode or "clarification / control flow")
    if responses.last_error:
        print("Generation error:", responses.last_error)
    if agent.action_runner.last_tool_result is not None:
        print("Tool:", agent.action_runner.last_tool_result.model_dump())
    print()
    return answer

agent.reset()
chat("Where is my order?")
chat("My order is ORD-123.")

In [ ]:
agent.reset()
chat("What time does the Central store close on Saturday?")
agent.reset()
chat("Does Central have the blue jacket in stock?")
agent.reset()
chat("Please cancel ORD-123.")

Expected behaviour: ask for the missing order ID; use the mock lookup to answer the next turn; retrieve Central's hours and answer 18:00 on Saturday; explicitly state that live stock is unknown; report cancellation as unsupported without executing or claiming it.

Guardrails depend on the analyzer choosing the correct intent. This integration does not make routing errors impossible. Pending order follow-ups still assume the user is supplying an ID; switching tasks during a pending action and complex knowledge follow-ups remain future work. Use `agent.reset()` between independent demonstrations.

For your own messages, run `chat("your question")`. Inspect `responses.last_messages` to see exactly what was passed to V2. End-to-end latency includes classification and a base-model analysis call as well as the final response, so it will exceed the generation-only benchmark.